# StyleFit AI - Exploratory Data Analysis & Visual Insights

## 1. Project Context & Pre-Purchase Machine Learning Constraint

**StyleFit AI** aims to predict clothing fit (`small`, `fit`, `large`) for Rent the Runway apparel rentals.

> **CRITICAL PRODUCTION CONSTRAINT (Data Leakage Protection)**:
> In a real-world pre-purchase recommendation system, customer reviews (`review_text`, `review_summary`) and star ratings (`rating`) are generated **post-purchase**. Therefore, these fields **MUST NOT** be used as feature inputs in production models or in production-oriented feature relationship analysis.
> Rating is included in this notebook strictly for descriptive missingness overview.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
current_dir = Path.cwd().resolve()
project_root = current_dir if (current_dir / "src").exists() else current_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.eda import load_dataset, parse_height_inches, parse_weight_lbs, get_parsing_report, run_eda_pipeline
from src.visualization import (
    plot_target_distribution,
    plot_missing_values,
    plot_age_distribution,
    plot_size_distribution,
    plot_physical_distributions,
    plot_category_distribution,
    plot_rented_for_body_type,
    plot_fit_by_categorical_factors,
    plot_user_item_sparsity,
)

## 2. Load Dataset & In-Memory Height/Weight Parsing

The dataset is loaded read-only from `data/raw/renttherunway_final_data.json.gz`.
Height and weight parsing is performed in-memory purely for descriptive visualization; the raw dataset is not modified.

In [2]:
df = load_dataset()
print(f"Raw Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")

h_parsed = parse_height_inches(df["height"])
w_parsed = parse_weight_lbs(df["weight"])

parse_report = get_parsing_report(df)
print("\n--- In-Memory Height/Weight Parsing Summary ---")
print(f"Height - Total: {parse_report['height']['total_non_null']:,}, Parsed: {parse_report['height']['parsed_successful']:,}, Failed: {parse_report['height']['parsed_failed']}")
print(f"Height Range: {int(parse_report['height']['min_inches'])} inches to {int(parse_report['height']['max_inches'])} inches")
print(f"Weight - Total: {parse_report['weight']['total_non_null']:,}, Parsed: {parse_report['weight']['parsed_successful']:,}, Failed: {parse_report['weight']['parsed_failed']}")
print(f"Weight Range: {int(parse_report['weight']['min_lbs'])} lbs to {int(parse_report['weight']['max_lbs'])} lbs")

Raw Dataset Shape: 192,544 rows, 15 columns



--- In-Memory Height/Weight Parsing Summary ---
Height - Total: 191,867, Parsed: 191,867, Failed: 0
Height Range: 54 inches to 78 inches
Weight - Total: 162,562, Parsed: 162,562, Failed: 0
Weight Range: 50 lbs to 300 lbs


## 3. Target Distribution (`fit`)

Visualizing class imbalance across the target categories: `fit` (73.78%), `small` (13.39%), and `large` (12.83%).

In [3]:
fig_path_1 = plot_target_distribution(df)
print(f"Saved Figure 1: {fig_path_1}")

Saved Figure 1: C:\Users\mahta\my_projects\stylefit-ai\reports\figures\01_target_distribution.png


## 4. Missing Value Patterns (NaN Percentage per Column)

Overview of missing value percentages per column.
- **Missingness Definition**: The chart displays NaN percentages across columns.
- **Whitespace-only Strings**: Evaluated separately. `review_text` contains 2 whitespace-only entries, and `review_summary` contains 16.
- **Data Leakage Boundary**: `rating`, `review_text`, and `review_summary` are explicitly highlighted as post-purchase fields.

In [4]:
fig_path_2 = plot_missing_values(df)
print(f"Saved Figure 2: {fig_path_2}")

Saved Figure 2: C:\Users\mahta\my_projects\stylefit-ai\reports\figures\02_missing_values.png


## 5. Customer Age Distribution

Matplotlib histogram focused on readable range 16-90 years, annotating missing ages (960), ages < 16 (90), and ages > 90 (87).
*Note: No dataset rows have been removed.*

In [5]:
fig_path_3 = plot_age_distribution(df)
print(f"Saved Figure 3: {fig_path_3}")

Saved Figure 3: C:\Users\mahta\my_projects\stylefit-ai\reports\figures\03_age_distribution.png


## 6. Selected Size Value Distribution & Full Size Table

Frequency distribution of selected size values using a logarithmic scale for legibility.
- **Neutral Size 0 Note**: Size value 0 is present in 526 rows and is not automatically treated as invalid or missing data.
- **Full Size Distribution Table**: Preserved below in full detail.

In [6]:
fig_path_4 = plot_size_distribution(df)
print(f"Saved Figure 4: {fig_path_4}")

print("\n--- Full Unfiltered Size Distribution Table ---")
size_table = pd.DataFrame({
    "count": df["size"].value_counts().sort_index(),
    "percentage": (df["size"].value_counts(normalize=True).sort_index() * 100).round(3)
})
print(size_table.to_string())

Saved Figure 4: C:\Users\mahta\my_projects\stylefit-ai\reports\figures\04_size_distribution.png

--- Full Unfiltered Size Distribution Table ---
      count  percentage
size                   
0       526       0.273
1     13719       7.125
2       729       0.379
3       711       0.369
4     29562      15.353
5      1739       0.903
6         9       0.005
7       673       0.350
8     40804      21.192
9      2651       1.377
10        3       0.002
11      437       0.227
12    24702      12.829
13     2573       1.336
14    11921       6.191
15      353       0.183
16    17668       9.176
17     2120       1.101
18        6       0.003
19      107       0.056
20    18500       9.608
21     1661       0.863
22       11       0.006
23       88       0.046
24     9261       4.810
25     1240       0.644
26      898       0.466
27       47       0.024
28     3488       1.812
29      522       0.271
30        6       0.003
32     1236       0.642
33       32       0.017
34       13    

## 7. Parsed Height & Weight Continuous Distributions

In [7]:
fig_path_5 = plot_physical_distributions(df, h_parsed, w_parsed)
print(f"Saved Figure 5: {fig_path_5}")

Saved Figure 5: C:\Users\mahta\my_projects\stylefit-ai\reports\figures\05_height_weight_distributions.png


## 8. Clothing Category Frequency & Long-Tail Distribution

In [8]:
fig_path_6 = plot_category_distribution(df)
print(f"Saved Figure 6: {fig_path_6}")

Saved Figure 6: C:\Users\mahta\my_projects\stylefit-ai\reports\figures\06_category_distribution.png


## 9. Rental Occasion & Body Type Distributions

In [9]:
fig_path_7 = plot_rented_for_body_type(df)
print(f"Saved Figure 7: {fig_path_7}")

Saved Figure 7: C:\Users\mahta\my_projects\stylefit-ai\reports\figures\07_rented_for_body_type.png


## 10. Fit Outcome Proportions & Trends by Pre-Purchase Categorical Factors

Within-group fit outcome proportions (`small`, `fit`, `large`) with minimum sample size thresholds:
- **Categories** (Horizontal 100% Stacked Bar): $N \ge 1,000$
- **Selected Size** (Unsmoothed Line Trend with Markers): $N \ge 500$. Markers represent actual observed size values with no line smoothing or interpolation.
- **Rental Occasions & Body Types** (Horizontal 100% Stacked Bar): $N \ge 100$

In [10]:
fig_path_8 = plot_fit_by_categorical_factors(df)
print(f"Saved Figure 8: {fig_path_8}")

Saved Figure 8: C:\Users\mahta\my_projects\stylefit-ai\reports\figures\08_fit_by_categorical_factors.png


## 11. User & Item Sparsity Distributions (Binned Counts)

Binned transaction frequencies (`1`, `2`, `3-5`, `6-10`, `11+`) across users and items.

In [11]:
fig_path_9 = plot_user_item_sparsity(df)
print(f"Saved Figure 9: {fig_path_9}")

Saved Figure 9: C:\Users\mahta\my_projects\stylefit-ai\reports\figures\09_user_item_sparsity.png


## 12. Key EDA Findings & Modeling Implications

1. **Class Imbalance**: Target `fit` is dominated by `fit` (73.78%), `small` (13.39%), `large` (12.83%). Requires stratified splitting and loss weighting.
2. **Data Leakage Protection**: `review_text`, `review_summary`, `rating` are post-purchase fields. Production features must rely solely on pre-purchase inputs.
3. **Missing Value Strategy**: Pre-purchase missingness (`weight`: 15.57%, `bust size`: 9.56%, `body type`: 7.60%) will require explicit missing indicators or median/mode imputation. Whitespace-only strings in reviews (2 text, 16 summary) are kept separate from NaNs.
4. **Category Aggregation**: 14 categories have < 10 rows and 37 have < 100 rows. Rare categories should be grouped into an `"other"` category during preprocessing.
5. **Selected Size**: Size value 0 is present (526 rows) and valid. Size trend plot shows stability across $N \ge 500$ sizes.
6. **Sparsity**: 68.0% of users (71,824) and 5.8% of items (341) have only 1 transaction. Body measurement features (height, weight, size, body type) will be vital for cold-start recommendations.